# 02 - Scheduled Briefings: promotion that curates, remembers where it came
from, and runs without you

Notebook 01 fixed how the harness learns *procedures*. This notebook fixes the
other promotion path - scratch notes graduating into long-term memory - which the
review summarised as *"promotion without curation, growth without forgetting"*:

| Review gap (Path 1) | This notebook |
|---|---|
| Everything graduates within the hour | **Opt-in** `/inbox` + an LLM **triage gate** (keep / discard / hold) |
| 200-word chunks re-extracted by the LLM | **Whole-note promotion**, verbatim (`extract_memories=False`) |
| No provenance, nothing supersedes | A **queryable sidecar** (source path, revision, supersession links) |
| Consumer never actually scheduled | In-DB **DBMS_SCHEDULER** producer + drainable consumer + **queue-depth health check** |
| (bonus) persistence-of-one-fact only | **Continuity of work**: a new session resumes the briefing *plan*, not just facts |

The payoff: a **morning asset-risk briefing** the database assembles on schedule -
the LLM narrates the numbers; it never invents them.

## 0 · Setup

Connect, verify notebook 01's artifacts, check privileges, stand up the embedder.

In [ ]:
import array
import json
import uuid

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model, litellm_spec
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import promote

settings = load_settings()
conn = get_connection(settings)
require(conn, "01_self_improving_copilot")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

In [ ]:
# The scheduled pipeline needs two privileges most ADB users lack by default.
with conn.cursor() as cur:
    cur.execute("SELECT privilege FROM user_sys_privs")
    _privs = {r[0] for r in cur.fetchall()}
_missing = {"CREATE PROCEDURE", "CREATE JOB"} - _privs
check(not _missing,
      f"scheduler privileges present (run as ADMIN if missing: "
      + "; ".join(f'GRANT {p} TO {settings.db_user}' for p in sorted(_missing)) + ")"
      if _missing else "scheduler privileges present (CREATE PROCEDURE, CREATE JOB)")

In [ ]:
import asyncio
import numpy as np
from langchain_oracledb.embeddings.oracleai import OracleEmbeddings
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name):
        self.provider = OracleEmbeddings(
            conn=conn, params={"provider": "database", "model": model_name}
        )

    def embed(self, texts, *, is_query=False):
        return np.asarray(self.provider.embed_documents(list(texts)), dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


embedder = OracleONNXEmbedder(conn, settings.embed_model)
check(embedder.embed(["hello"]).shape == (1, 384), "in-database embedder ready")

## 1 · The Agent Memory SDK - configured for *whole-note* promotion

`extract_memories=False` is a deliberate review fix, not a shortcut: the original
pipeline chunked files into 200-word windows and had the LLM *re-interpret* each one
at ingest - confident misreadings of mid-sentence fragments became durable facts.
Here a curated note is stored **verbatim** as one memory; the triage gate (next
section) is where judgement happens, *before* anything is durable.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory, SchemaPolicy
from oracleagentmemory.core.llms.llm import Llm as SdkLlm

memory = OracleAgentMemory(
    connection=conn,
    embedder=embedder,
    llm=SdkLlm(**litellm_spec(settings)),
    extract_memories=False,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="CITY_",
)

# Round-trip smoke test: store, find, delete.
_mid = memory.add_memory("smoke test memory - safe to delete",
                         memory_type="fact", user_id="_smoke", agent_id="CITY")
_hits = memory.search("smoke test", user_id="_smoke", agent_id="CITY", max_results=1)
check(len(_hits) == 1 and _hits[0].id == _mid, "SDK add_memory/search round-trip")
memory.delete_memory(_mid)
ok("OracleAgentMemory ready (CITY_ tables, whole-note mode)")

## 2 · The scratchpad: cheap to write, safe to be wrong in

A plain table posing as a tiny filesystem. Inspectors jot notes all day; only
`/inbox/<asset>/...` is a promotion *candidate area*. Working files - drafts,
abandoned queries, tool dumps - live outside it and are **never** promoted, which
restores the scratchpad's contract the review showed the old pipeline destroyed.

In [ ]:
def _table_exists(name):
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = :t", t=name)
        return cur.fetchone()[0] > 0


if not _table_exists("HARNESS_SCRATCH"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_SCRATCH (
            path       VARCHAR2(512) PRIMARY KEY,
            content    CLOB NOT NULL,
            revision   VARCHAR2(64) NOT NULL,
            status     VARCHAR2(1) DEFAULT 'N'
                       CHECK (status IN ('N','S','P','D','H')),
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
    conn.commit()


def write_note(path, content):
    rev = promote.note_revision(content)
    with conn.cursor() as cur:
        cur.execute("DELETE FROM HARNESS_SCRATCH WHERE path = :p", p=path)
        cur.execute("""INSERT INTO HARNESS_SCRATCH (path, content, revision, status)
                       VALUES (:p, :c, :r, 'N')""", p=path, c=content, r=rev)
    conn.commit()
    return rev


def read_note(path):
    with conn.cursor() as cur:
        cur.execute("SELECT content, revision, status FROM HARNESS_SCRATCH WHERE path = :p",
                    p=path)
        row = cur.fetchone()
    if row is None:
        return None
    content = row[0].read() if hasattr(row[0], "read") else row[0]
    return {"path": path, "content": content, "revision": row[1], "status": row[2]}


def list_notes(prefix="/"):
    with conn.cursor() as cur:
        cur.execute("""SELECT path, status FROM HARNESS_SCRATCH
                       WHERE path LIKE :p ORDER BY path""", p=prefix + "%")
        return cur.fetchall()


def set_note_status(path, status):
    with conn.cursor() as cur:
        cur.execute("UPDATE HARNESS_SCRATCH SET status = :s WHERE path = :p",
                    s=status, p=path)
    conn.commit()

ok("scratch store ready")

In [ ]:
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
_asset_names = sorted({x["asset_name"] for x in _logs})
ASSET_A = "Harbor Bridge" if "Harbor Bridge" in _asset_names else _asset_names[0]

# Seed the inbox with real field narratives (non-routine first - richer signal),
# and the review's own cautionary trio OUTSIDE the inbox.
_candidates = [l for l in _logs if l["asset_name"] == ASSET_A and l["severity"] != "routine"]
_candidates += [l for l in _logs if l["asset_name"] == ASSET_A and l["severity"] == "routine"]
INBOX_SEED = []
for i, log in enumerate(_candidates[:4]):
    path = f"/inbox/{ASSET_A}/{i}.md"
    INBOX_SEED.append((path, log["narrative"], log["days_ago"]))
    write_note(path, log["narrative"])

SUPERSEDE_DEFECT_PATH = f"/inbox/{ASSET_A}/defect-bearing.md"
SUPERSEDE_REPAIR_PATH = f"/inbox/{ASSET_A}/repair-bearing.md"
write_note(SUPERSEDE_DEFECT_PATH,
           f"Bearing plates at pier 2 of {ASSET_A} show active corrosion with "
           "measurable section loss. Load rating review recommended; monitor monthly.")

write_note("/work/draft_notes.md",
           "TODO: check if the load rating includes the retrofit?? probably not")
write_note("/work/attempt1_wrong.sql",
           "SELECT asset, COUNT(*) FROM findings GROUP BY severity  -- wrong, abandoned")
write_note("/tool_out/similar_dump.json", json.dumps({"rows": list(range(50))}))

for p, s in list_notes("/"):
    print(f"{s}  {p}")
ok(f"{len(INBOX_SEED) + 1} inbox candidates, 3 working files (outside the inbox)")

## 3 · The triage gate: judgement *before* durability

Two filters, in order:

1. **Opt-in area** - anything outside `/inbox/` is discarded without spending a
   model call (the review's `/work/attempt1_wrong.sql` can never leak into memory)
2. **LLM triage** - inbox notes are classified `keep` (durable, factual, worth
   retrieval attention), `discard` (ephemeral, speculative, duplicative), or
   `hold` (incomplete - stays in scratch, re-triaged next cycle)

In [ ]:
from pydantic import BaseModel
from typing import Literal


class Triage(BaseModel):
    decision: Literal["keep", "discard", "hold"]
    reason: str


_triage_model = CHAT.with_structured_output(Triage)

TRIAGE_PROMPT = (
    "You are the curation gate for a city-infrastructure agent's long-term memory.\n"
    "Everything stored durably competes for retrieval attention forever, so be picky.\n"
    "keep    = a completed observation or action worth recalling months from now\n"
    "          (equipment facts, completed work, hazards, measured conditions).\n"
    "discard = ephemeral, speculative, draft-like, or content-free.\n"
    "hold    = potentially valuable but incomplete or unresolved (open questions,\n"
    "          pending measurements).\n\n"
    "Note path: {path}\nNote content:\n{content}"
)


def triage_note(path, content):
    # ✏️ TODO(1): the opt-in gate comes FIRST and never costs a model call.
    #             If promote.triage_allowed(path) is False, return
    #             Triage(decision='discard', reason='outside the /inbox opt-in area').
    # TODO-SOLUTION-START
    if not promote.triage_allowed(path):
        return Triage(decision="discard", reason="outside the /inbox opt-in area")
    # TODO-SOLUTION-END
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    return _triage_model.invoke(TRIAGE_PROMPT.format(path=path, content=content), config=cfg)


for p, _s in list_notes("/"):
    note = read_note(p)
    t = triage_note(p, note["content"])
    print(f"{t.decision:8s} {p}\n         {t.reason[:100]}")
ok("triage gate live")

## 4 · Promotion: whole notes, with a paper trail

A kept note is stored **verbatim** as one memory. Its provenance - source path,
content revision, observation age - lands in a **sidecar table** you can query with
SQL, not a blob you can't. And before storing, a supersession check asks: does this
note *replace* something we already believe? (A repair note supersedes the defect it
fixed - the review's 'archaeology site' problem, closed.)

In [ ]:
if not _table_exists("HARNESS_MEMORY_META"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_MEMORY_META (
            memory_id     VARCHAR2(128) PRIMARY KEY,
            asset_id      VARCHAR2(128) NOT NULL,
            source_path   VARCHAR2(512) NOT NULL,
            revision      VARCHAR2(64) NOT NULL,
            days_ago      NUMBER DEFAULT 0,
            superseded_by VARCHAR2(128),
            promoted_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
    conn.commit()


def asset_from_path(path):
    parts = path.split("/")
    return parts[2] if len(parts) > 3 and parts[1] == "inbox" else None


class Supersedes(BaseModel):
    supersedes: bool
    reason: str


_supersede_model = CHAT.with_structured_output(Supersedes)


def _check_supersession(content, candidate):
    """candidate: (memory_id, existing_content). True if the new note makes the
    old one obsolete (repair completes a defect, a measurement replaces an estimate)."""
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    q = ("Existing memory:\n{old}\n\nNew note:\n{new}\n\n"
         "Does the new note make the existing memory OBSOLETE (completed repair,\n"
         "corrected fact, replaced measurement)? Additional detail alone does not\n"
         "supersede.").format(old=candidate[1], new=content)
    return _supersede_model.invoke(q, config=cfg)


def promote_note(path):
    note = read_note(path)
    t = triage_note(path, note["content"])
    if t.decision == "discard":
        set_note_status(path, "D")
        return None
    if t.decision == "hold":
        set_note_status(path, "H")
        return None
    asset_id = asset_from_path(path)
    # ✏️ TODO(2): supersession check - search this asset's existing memories
    #             (memory.search(note content, user_id=asset_id, agent_id='CITY',
    #             max_results=3)), skip any already superseded (sidecar lookup),
    #             and collect ids the LLM says are made obsolete.
    # TODO-SOLUTION-START
    obsolete = []
    for hit in memory.search(note["content"], user_id=asset_id, agent_id="CITY",
                             max_results=3):
        with conn.cursor() as cur:
            cur.execute("""SELECT superseded_by FROM HARNESS_MEMORY_META
                           WHERE memory_id = :m""", m=hit.id)
            row = cur.fetchone()
        if row and row[0]:
            continue  # already superseded - not a candidate
        verdict = _check_supersession(note["content"], (hit.id, hit.content))
        if verdict.supersedes:
            obsolete.append(hit.id)
            print(f"  supersedes {hit.id[:12]}...: {verdict.reason[:80]}")
    # TODO-SOLUTION-END
    # ✏️ TODO(3): whole-note promotion - store the note VERBATIM as one memory
    #             (memory.add_memory, memory_type='fact', user_id=asset_id,
    #             agent_id='CITY'), record provenance in the sidecar, link
    #             superseded ids, and mark the scratch note 'P'.
    # TODO-SOLUTION-START
    mid = memory.add_memory(note["content"], memory_type="fact",
                            user_id=asset_id, agent_id="CITY")
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO HARNESS_MEMORY_META
                       (memory_id, asset_id, source_path, revision)
                       VALUES (:m, :a, :p, :r)""",
                    m=mid, a=asset_id, p=path, r=note["revision"])
        for old_id in obsolete:
            cur.execute("""UPDATE HARNESS_MEMORY_META SET superseded_by = :new
                           WHERE memory_id = :old""", new=mid, old=old_id)
    conn.commit()
    set_note_status(path, "P")
    # TODO-SOLUTION-END
    return mid

ok("promotion pipeline defined (triage -> supersession -> whole-note store)")

## 5 · Actually scheduled: producer in the database, consumer with a pulse

The review's sharpest Path-1 finding: the producer was a real scheduled job but the
consumer only ever ran when a human ran a cell - *"the pipeline silently half-works,
which is worse than visibly failing."* Here:

- the **producer** (`HARNESS_STAGE_JOB`, hourly) is pure SQL inside the database -
  it stages new inbox notes into a queue whether or not any Python process exists
- the **consumer** (`drain_queue`) runs the triage/promotion pipeline; in production
  your orchestrator schedules it - and because that can fail, the
- **health check** turns queue depth + age into an ok/warn/stalled status a
  dashboard can alert on. A stalled pipeline is *visible*.

In [ ]:
if not _table_exists("HARNESS_PROMOTION_QUEUE"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_PROMOTION_QUEUE (
            queue_id    VARCHAR2(64) PRIMARY KEY,
            path        VARCHAR2(512) NOT NULL,
            enqueued_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            consumed    VARCHAR2(1) DEFAULT 'N' CHECK (consumed IN ('N','Y')))""")
    conn.commit()

with conn.cursor() as cur:
    cur.execute("""CREATE OR REPLACE PROCEDURE HARNESS_STAGE_INBOX AS
    BEGIN
      INSERT INTO harness_promotion_queue (queue_id, path)
      SELECT RAWTOHEX(SYS_GUID()), s.path
        FROM harness_scratch s
       WHERE s.status = 'N' AND s.path LIKE '/inbox/%'
         AND NOT EXISTS (SELECT 1 FROM harness_promotion_queue q
                          WHERE q.path = s.path AND q.consumed = 'N');
      UPDATE harness_scratch SET status = 'S'
       WHERE status = 'N' AND path LIKE '/inbox/%';
      COMMIT;
    END;""")
ok("producer procedure HARNESS_STAGE_INBOX compiled")

In [ ]:
def _replace_job(name, action, repeat, start_offset_minutes=1):
    with conn.cursor() as cur:
        try:
            cur.callproc("DBMS_SCHEDULER.DROP_JOB", [name])
        except Exception:
            pass
        cur.execute("""BEGIN
            DBMS_SCHEDULER.CREATE_JOB(
                job_name        => :name,
                job_type        => 'STORED_PROCEDURE',
                job_action      => :action,
                start_date      => SYSTIMESTAMP + NUMTODSINTERVAL(:offs, 'MINUTE'),
                repeat_interval => :rep,
                enabled         => TRUE);
        END;""", name=name, action=action, rep=repeat, offs=start_offset_minutes)
    conn.commit()


_replace_job("HARNESS_STAGE_JOB", "HARNESS_STAGE_INBOX", "FREQ=HOURLY")
with conn.cursor() as cur:
    cur.execute("""SELECT job_name, enabled, repeat_interval
                     FROM user_scheduler_jobs WHERE job_name = 'HARNESS_STAGE_JOB'""")
    print(cur.fetchone())
# Fire it now rather than waiting an hour - same code path the schedule runs.
with conn.cursor() as cur:
    cur.callproc("DBMS_SCHEDULER.RUN_JOB", ["HARNESS_STAGE_JOB", True])
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM HARNESS_PROMOTION_QUEUE WHERE consumed = 'N'")
    _pending = cur.fetchone()[0]
check(_pending >= 5, f"producer staged {_pending} inbox notes into the queue")

In [ ]:
def drain_queue(limit=50):
    """The consumer: triage -> supersession -> whole-note promotion, per queue item.
    In production an orchestrator runs this on its own schedule."""
    with conn.cursor() as cur:
        cur.execute("""SELECT queue_id, path FROM HARNESS_PROMOTION_QUEUE
                       WHERE consumed = 'N' ORDER BY enqueued_at
                       FETCH FIRST :n ROWS ONLY""", n=limit)
        items = cur.fetchall()
    done = 0
    for qid, path in items:
        mid = promote_note(path)
        with conn.cursor() as cur:
            cur.execute("UPDATE HARNESS_PROMOTION_QUEUE SET consumed = 'Y' "
                        "WHERE queue_id = :q", q=qid)
        conn.commit()
        done += 1
        note_status = read_note(path)["status"]
        print(f"  {note_status}  {path}" + (f"  -> memory {mid[:12]}..." if mid else ""))
    return done


def pipeline_health():
    with conn.cursor() as cur:
        cur.execute("""SELECT COUNT(*),
                              (CAST(CURRENT_TIMESTAMP AS DATE)
                               - CAST(MIN(enqueued_at) AS DATE)) * 24 * 60
                         FROM HARNESS_PROMOTION_QUEUE WHERE consumed = 'N'""")
        pending, oldest_minutes = cur.fetchone()
    # ✏️ TODO(4): turn raw pending/oldest-age numbers into an alertable status
    #             with promote.queue_health(pending, oldest_minutes).
    # TODO-SOLUTION-START
    status = promote.queue_health(pending, oldest_minutes)
    # TODO-SOLUTION-END
    print(f"queue: pending={pending} oldest={oldest_minutes or 0:.0f}min -> {status}")
    return status


check(pipeline_health() in ("ok", "warn"), "health visible before draining")
print(f"drained {drain_queue()} items")
check(pipeline_health() == "ok", "queue drained - pipeline healthy")

In [ ]:
# The scratchpad contract, restored: working files untouched, inbox curated.
for p, s in list_notes("/"):
    print(f"{s}  {p}")
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM HARNESS_MEMORY_META")
    _promoted = cur.fetchone()[0]
with conn.cursor() as cur:
    cur.execute("""SELECT status, COUNT(*) FROM HARNESS_SCRATCH
                   WHERE path NOT LIKE '/inbox/%' GROUP BY status""")
    _outside = dict(cur.fetchall())
check(_promoted >= 3, f"{_promoted} whole notes promoted with provenance")
check(set(_outside) <= {"N", "D"}, "nothing outside /inbox was staged or promoted")

## 6 · The morning briefing: the database computes, the model narrates

`HARNESS_BUILD_BRIEFING` runs daily at 06:00 **inside the database** - it aggregates
recent findings, queue health, and memory counts into a JSON payload with plain SQL.
No LLM in the loop, so it cannot hallucinate a number. The narrative layer runs when
a human reads the briefing - grounded, by instruction, in only the JSON's numbers.

In [ ]:
if not _table_exists("HARNESS_BRIEFING"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_BRIEFING (
            briefing_id  VARCHAR2(64) PRIMARY KEY,
            generated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            payload      CLOB NOT NULL,
            narrative    CLOB)""")
    conn.commit()

with conn.cursor() as cur:
    cur.execute("""CREATE OR REPLACE PROCEDURE HARNESS_BUILD_BRIEFING AS
      payload CLOB;
    BEGIN
      SELECT JSON_OBJECT(
        'generated_for' VALUE TO_CHAR(SYSDATE, 'YYYY-MM-DD'),
        'queue_pending' VALUE (SELECT COUNT(*) FROM harness_promotion_queue
                                WHERE consumed = 'N'),
        'curated_memories' VALUE (SELECT COUNT(*) FROM harness_memory_meta
                                   WHERE superseded_by IS NULL),
        'recent_findings' VALUE (
            SELECT JSON_ARRAYAGG(JSON_OBJECT('asset' VALUE asset_id,
                                             'severity' VALUE severity,
                                             'cnt' VALUE n) RETURNING CLOB)
              FROM (SELECT asset_id, severity, COUNT(*) AS n
                      FROM city_inspection_finding
                     WHERE days_ago <= 30
                     GROUP BY asset_id, severity
                     ORDER BY n DESC FETCH FIRST 10 ROWS ONLY))
        RETURNING CLOB) INTO payload FROM dual;
      INSERT INTO harness_briefing (briefing_id, payload)
      VALUES (RAWTOHEX(SYS_GUID()), payload);
      COMMIT;
    END;""")

_replace_job("HARNESS_BRIEFING_JOB", "HARNESS_BUILD_BRIEFING",
             "FREQ=DAILY;BYHOUR=6;BYMINUTE=0")
with conn.cursor() as cur:
    cur.callproc("DBMS_SCHEDULER.RUN_JOB", ["HARNESS_BRIEFING_JOB", True])
ok("briefing job scheduled daily at 06:00 - and fired once, now")

In [ ]:
def latest_briefing():
    with conn.cursor() as cur:
        cur.execute("""SELECT briefing_id, generated_at, payload, narrative
                         FROM HARNESS_BRIEFING ORDER BY generated_at DESC
                        FETCH FIRST 1 ROWS ONLY""")
        row = cur.fetchone()
    if row is None:
        return None
    payload = row[2].read() if hasattr(row[2], "read") else row[2]
    narrative = row[3].read() if row[3] is not None and hasattr(row[3], "read") else row[3]
    return {"briefing_id": row[0], "generated_at": row[1],
            "payload": json.loads(payload), "narrative": narrative}


def narrate_briefing():
    b = latest_briefing()
    # ✏️ TODO(5): the grounded narration - ask CHAT for a <=6-sentence morning
    #             briefing using ONLY numbers present in the JSON payload, then
    #             write the narrative back onto the briefing row.
    # TODO-SOLUTION-START
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    resp = CHAT.invoke(
        "Write a morning asset-risk briefing for a city operations chief.\n"
        "HARD RULE: use ONLY facts and numbers present in this JSON - do not\n"
        "estimate, extrapolate, or invent. <= 6 sentences, plain prose.\n\n"
        + json.dumps(b["payload"]), config=cfg)
    text = resp.content if isinstance(resp.content, str) else "".join(
        p.get("text", "") if isinstance(p, dict) else str(p) for p in resp.content)
    with conn.cursor() as cur:
        cur.execute("UPDATE HARNESS_BRIEFING SET narrative = :n WHERE briefing_id = :b",
                    n=text, b=b["briefing_id"])
    conn.commit()
    # TODO-SOLUTION-END
    return text


b = latest_briefing()
check(b is not None and b["payload"]["curated_memories"] >= 3,
      f"briefing payload built in-database: {json.dumps(b['payload'])[:120]}...")
print()
print(narrate_briefing())

## 7 · Supersession in action, and recall that prefers the present

The repair note arrives, gets staged by the *scheduled* producer, drained, and -
because the LLM confirms it completes the defect - the old memory is marked
superseded in the sidecar. Recall then: filters superseded memories out, joins
provenance back in, and recency-weights what remains.

In [ ]:
write_note(SUPERSEDE_REPAIR_PATH,
           f"Bearing plates at pier 2 of {ASSET_A} replaced and recoated on today's"
           " shift; corrosion remediated, section loss addressed. Load rating"
           " restored to design values. Next scheduled check in 12 months.")
with conn.cursor() as cur:
    cur.callproc("DBMS_SCHEDULER.RUN_JOB", ["HARNESS_STAGE_JOB", True])
print(f"drained {drain_queue()} item(s)")
with conn.cursor() as cur:
    cur.execute("""SELECT COUNT(*) FROM HARNESS_MEMORY_META m
                    JOIN HARNESS_SCRATCH s ON s.path = m.source_path
                   WHERE m.source_path = :p AND m.superseded_by IS NOT NULL""",
                p=SUPERSEDE_DEFECT_PATH)
    _superseded = cur.fetchone()[0]
check(_superseded == 1, "defect memory marked superseded by the repair note")

In [ ]:
def recall(query, asset_id, k=5):
    """SDK relevance search -> sidecar join (drop superseded, attach provenance)
    -> recency re-rank."""
    hits = memory.search(query, user_id=asset_id, agent_id="CITY", max_results=k * 2)
    rows = []
    for idx, h in enumerate(hits):
        with conn.cursor() as cur:
            cur.execute("""SELECT source_path, revision, days_ago, superseded_by
                             FROM HARNESS_MEMORY_META WHERE memory_id = :m""", m=h.id)
            meta = cur.fetchone()
        if meta is None or meta[3] is not None:
            continue  # unprovenanced or superseded
        rows.append({"memory_id": h.id, "content": h.content,
                     "source_path": meta[0], "revision": meta[1][:12],
                     "days_ago": meta[2] or 0, "base": 1.0 / (1 + idx)})
    return promote.rerank_by_recency(rows)[:k]


results = recall("bearing plate corrosion at pier 2", ASSET_A)
for r in results:
    print(f"score={r['score']:.3f} days_ago={r['days_ago']:>4} {r['source_path']}\n"
          f"    {r['content'][:100]}...")
_texts = " ".join(r["content"] for r in results)
check("replaced" in _texts or "remediated" in _texts, "repair note is recalled")
check(all(r["source_path"] != SUPERSEDE_DEFECT_PATH for r in results),
      "superseded defect memory no longer competes for retrieval attention")

## 8 · Continuity: a new session resumes the *work*, not just the facts

The review granted the original notebook fact-persistence but noted the hard half
is resuming *work* - goal, done, next. So the briefing workflow keeps its plan as a
scratch note, and a brand-new connection + brand-new SDK client picks it up and
continues, without this notebook's kernel state.

In [ ]:
write_note("/plans/briefing_plan.md", json.dumps({
    "goal": "daily 06:00 asset-risk briefing, grounded narration on read",
    "done": ["producer + consumer + health check", "briefing job scheduled",
             "first briefing narrated"],
    "next": "review narration tone with operations chief; tune half-life",
}))
ok("briefing plan state persisted to /plans/briefing_plan.md")

In [ ]:
# A second 'session': fresh connection, fresh SDK client - no shared kernel state.
conn2 = get_connection(settings)
memory2 = OracleAgentMemory(
    connection=conn2,
    embedder=OracleONNXEmbedder(conn2, settings.embed_model),
    llm=SdkLlm(**litellm_spec(settings)),
    extract_memories=False,
    schema_policy=SchemaPolicy.REQUIRE_EXISTING,
    table_name_prefix="CITY_",
)

with conn2.cursor() as cur:
    cur.execute("SELECT content FROM HARNESS_SCRATCH WHERE path = '/plans/briefing_plan.md'")
    _raw = cur.fetchone()[0]
plan = json.loads(_raw.read() if hasattr(_raw, "read") else _raw)
print("resumed goal:", plan["goal"])
print("next action :", plan["next"])

_hits2 = memory2.search("bearing plate repair", user_id=ASSET_A, agent_id="CITY",
                        max_results=2)
with conn2.cursor() as cur:
    cur.execute("""SELECT COUNT(*) FROM HARNESS_BRIEFING WHERE narrative IS NOT NULL""")
    _narrated = cur.fetchone()[0]
check(len(plan["next"]) > 0 and len(_hits2) > 0 and _narrated >= 1,
      "new session resumed the plan, the memories, and the briefing")
conn2.close()

## 9 · What you built (vs. what the review criticised)

| Review gap (Path 1) | Fix you built |
|---|---|
| Every scratch file graduates | `/inbox` opt-in + LLM triage (keep/discard/hold) |
| Chunked re-extraction invents facts | Whole-note promotion, verbatim |
| No provenance, no supersession | `HARNESS_MEMORY_META`: source path, revision, superseded_by - all SQL-queryable |
| Consumer never scheduled, failure silent | In-DB producer job + drainable consumer + ok/warn/stalled health |
| Nothing forgets | Superseded memories filtered out of recall; recency half-life weighting |
| Facts persist, work doesn't | Plan state resumed by a fresh session |

The jobs stay scheduled after you close this notebook - that is the point. Disable
them any time with `DBMS_SCHEDULER.DISABLE('HARNESS_STAGE_JOB')` /
`DBMS_SCHEDULER.DISABLE('HARNESS_BRIEFING_JOB')`.

Next: **03 - Context engineering**, where compaction and offloading get measured for
*fidelity*, not just size.

In [ ]:
for desc, passed in verify(conn, "02_scheduled_briefings"):
    check(passed, desc)
ok("notebook 02 artifacts in place - continue to 03_context_engineering")